# Análisis de ventas con PySpark

El análisis usa un renglón por factura y artículo. Agrupa las unidades y el importe por vendedor y producto para apoyar las asignaciones, y por vendedor y ciudad del cliente para planificar giras. El importe vendido es el criterio principal de los rankings y las unidades se muestran como referencia.


1. En SSMS, ejecutá el script `Exportar_ventas_para_Spark.sql` sobre las bases restauradas. Guardá la grilla como `ventas_spark.csv` con los encabezados. La opción **Incluir encabezados de columna al copiar o guardar los resultados** está en **Herramientas → Opciones → Resultados de consulta → SQL Server → Resultados en cuadrícula**. Si la cambiás, abrí una nueva ventana de consulta antes de exportar.
2. Subí el CSV a la sesión de Colab o guardalo junto a este notebook. La consulta de extracción conserva el detalle. Las agrupaciones y los rankings se hacen en Spark.
3. Ejecutá las celdas en orden. En Colab, si falta PySpark, descomentá la línea de instalación.

In [ ]:
# %pip install pyspark==3.5.6
from pathlib import Path

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = SparkSession.builder.master("local[*]").appName("TP3 Distribuidora Mayorista").getOrCreate()

ruta_csv = Path("ventas_spark.csv")
separador = ","  # Si el archivo se exportó con punto y coma, usar ";".
codificacion = "UTF-8"  # Si se exportó como Unicode de Windows, probar "UTF-16LE".
if not ruta_csv.is_file():
    raise FileNotFoundError("Subí ventas_spark.csv o cambiá ruta_csv.")


## Carga y controles del origen

Se revisa que cada renglón tenga sus identificadores, fecha y medidas. Si se repite una combinación de factura y artículo, se debe revisar el origen antes de sumar los importes.


In [ ]:
esquema = T.StructType([
    T.StructField("NumeroFactura", T.IntegerType()),
    T.StructField("IdArticulo", T.IntegerType()),
    T.StructField("IdVendedor", T.IntegerType()),
    T.StructField("Vendedor", T.StringType()),
    T.StructField("IdProducto", T.IntegerType()),
    T.StructField("Producto", T.StringType()),
    T.StructField("IdCiudad", T.IntegerType()),
    T.StructField("Ciudad", T.StringType()),
    T.StructField("Departamento", T.StringType()),
    T.StructField("Fecha", T.DateType()),
    T.StructField("Unidades", T.IntegerType()),
    T.StructField("Importe", T.DecimalType(18, 2)),
])

ventas = (
    spark.read.schema(esquema)
    .options(header=True, sep=separador, encoding=codificacion, dateFormat="yyyy-MM-dd",
             enforceSchema=False, mode="FAILFAST")
    .csv(str(ruta_csv))
    .cache()
)

obligatorias = ["NumeroFactura", "IdArticulo", "IdVendedor", "IdProducto", "IdCiudad",
                "Fecha", "Unidades", "Importe"]
faltantes = ventas.agg(*[
    F.count(F.when(F.col(columna).isNull(), 1)).alias(columna)
    for columna in obligatorias
]).first().asDict()
if any(faltantes.values()):
    raise ValueError(f"Revisá los datos faltantes en el CSV: {faltantes}")

duplicados = ventas.groupBy("NumeroFactura", "IdArticulo").count().where(F.col("count") > 1)
if duplicados.take(1):
    duplicados.show(truncate=False)
    raise ValueError("Hay más de una fila por factura y artículo.")

if not ventas.take(1):
    raise ValueError("El CSV no contiene ventas.")

ventas.agg(F.count("*").alias("Renglones"), F.countDistinct("NumeroFactura").alias("Facturas"),
           F.min("Fecha").alias("PrimeraFecha"), F.max("Fecha").alias("UltimaFecha"),
           F.sum("Unidades").alias("Unidades"), F.sum("Importe").alias("Importe")).show()


## Ventas por vendedor y producto

Se agrupan las distintas presentaciones de un artículo bajo su producto. El puesto indica qué vendedores registraron mayor importe para cada producto. Si hay empate en el importe, comparten puesto.


In [ ]:
por_producto = (
    ventas.groupBy("IdVendedor", "IdProducto")
    .agg(F.first("Vendedor", ignorenulls=True).alias("Vendedor"),
         F.first("Producto", ignorenulls=True).alias("Producto"),
         F.sum("Unidades").alias("UnidadesVendidas"),
         F.sum("Importe").alias("ImporteVendido"))
    .withColumn("PuestoEnProducto", F.dense_rank().over(
        Window.partitionBy("IdProducto").orderBy(F.desc("ImporteVendido"))
    ))
)

por_producto.select("Producto", "IdProducto", "Vendedor", "IdVendedor",
                    "UnidadesVendidas", "ImporteVendido", "PuestoEnProducto").orderBy(
    "IdProducto", "PuestoEnProducto", "IdVendedor"
).show(50, truncate=False)


## Ventas por vendedor y ciudad

El puesto indica qué vendedores registraron mayor importe en cada ciudad. Se agrupa por el identificador de ciudad para mantener diferenciados los registros del sistema que tienen el mismo nombre.


In [ ]:
por_ciudad = (
    ventas.groupBy("IdVendedor", "IdCiudad")
    .agg(F.first("Vendedor", ignorenulls=True).alias("Vendedor"),
         F.first("Ciudad", ignorenulls=True).alias("Ciudad"),
         F.first("Departamento", ignorenulls=True).alias("Departamento"),
         F.sum("Unidades").alias("UnidadesVendidas"),
         F.sum("Importe").alias("ImporteVendido"))
    .withColumn("PuestoEnCiudad", F.dense_rank().over(
        Window.partitionBy("IdCiudad").orderBy(F.desc("ImporteVendido"))
    ))
)

por_ciudad.select("Departamento", "Ciudad", "IdCiudad", "Vendedor", "IdVendedor",
                  "UnidadesVendidas", "ImporteVendido", "PuestoEnCiudad").orderBy(
    "IdCiudad", "PuestoEnCiudad", "IdVendedor"
).show(50, truncate=False)


## Control de totales

Los dos análisis parten del mismo conjunto de ventas. Por eso, al sumar todas las filas de cualquiera de los dos resultados, deben coincidir las unidades y el importe del origen.


In [ ]:
total_origen = ventas.agg(F.sum("Unidades"), F.sum("Importe")).first()
for nombre, resultado in [("Producto", por_producto), ("Ciudad", por_ciudad)]:
    total_resultado = resultado.agg(F.sum("UnidadesVendidas"), F.sum("ImporteVendido")).first()
    if tuple(total_resultado) != tuple(total_origen):
        raise ValueError(f"Los totales de {nombre} difieren del origen.")
    print(f"{nombre}: unidades e importe coinciden con el origen.")


Los puestos permiten identificar candidatos para asignar productos y ciudades según las ventas observadas en el período exportado. La cantidad de unidades ayuda a interpretar empates o diferencias de precio entre productos.
